# 08 — Figures 1, 2, 3 and 5

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### Figure 1 -- study design and analytical pipeline (Stages A-D).

In [ ]:
"""
Figure 1 -- study design and analytical pipeline (Stages A-D).
Run:     python make_figure1.py
Output:  Downloads/TCGA_GBM_STAGE_D_SURVIVAL/FIGURES/Figure1.png (600 dpi)
All text is in the BOXES dictionary-like calls below; edit wording there.
"""
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

OUT = BASE / "TCGA_GBM_STAGE_D_SURVIVAL" / "FIGURES"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7.5})
fig, ax = plt.subplots(figsize=(7.4, 6.4))
ax.set_xlim(0, 102); ax.set_ylim(0, 100); ax.axis("off")

INK = "#222222"
DATA_FC, DATA_EC = "#F2F2F2", "#7A7A7A"
STAGE = {  # header colour, box fill
    "A": ("#3F7A4E", "#EAF3EC"),
    "B": ("#2F6E8E", "#E6F0F5"),
    "C": ("#7A4E8C", "#F1EAF4"),
    "D": ("#3A5A80", "#E8EEF6"),
}
EXT_FC, EXT_EC = "#F7F2EA", "#9A7B4F"


def box(x, y, w, h, text, fc, ec, size=7.2, bold=False, color=INK):
    """x, y = centre."""
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                boxstyle="round,pad=0.2,rounding_size=0.9",
                                fc=fc, ec=ec, lw=0.9))
    ax.text(x, y, text, ha="center", va="center", fontsize=size, color=color,
            fontweight="bold" if bold else "normal", linespacing=1.25)


def arrow(x0, y0, x1, y1, color="#444444", lw=0.9, style="-|>"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle=style, lw=lw, color=color,
                                shrinkA=0, shrinkB=0))


# ------------------------------------------------------------------ inputs
ax.text(0.5, 98, "Data sources", fontsize=8.2, fontweight="bold", va="center")
DX = [10, 30, 50, 70, 90]
inputs = [
    "TCGA-GBM discovery\ntumours (recount3)\nn = 60",
    "GTEx cortex + BA9\n(recount3)\nn = 334 donors",
    "STRING protein\ninteractions",
    "TCGA-GBM survival\n(GDC STAR-Counts)\nn = 144 IDH-WT",
    "CGGA mRNAseq_693\nn = 104 IDH-WT\n(external)",
]
for x, t in zip(DX, inputs):
    box(x, 90, 18, 7.5, t, DATA_FC, DATA_EC, size=6.5)

# ------------------------------------------------------------------ stage columns
cols = {"A": 12.5, "B": 37.5, "C": 62.5, "D": 87.5}
W = 23
titles = {
    "A": "Stage A\nDisease anchors",
    "B": "Stage B\nReference network",
    "C": "Stage C\nPatient representations",
    "D": "Stage D\nSurvival modelling",
}
for k, x in cols.items():
    ec, _ = STAGE[k]
    box(x, 78, W, 5.5, titles[k], ec, ec, size=7.0, bold=True, color="white")
for a, b in (("A", "B"), ("B", "C"), ("C", "D")):
    arrow(cols[a] + W / 2 + 0.2, 78, cols[b] - W / 2 - 0.2, 78, "#444444", lw=1.0)

Y = [67.5, 56, 44.5, 33]
H = 8.6


def stage_boxes(k, texts):
    ec, fc = STAGE[k]
    x = cols[k]
    arrow(x, 78 - 2.75, x, Y[0] + H / 2, ec)
    for i, t in enumerate(texts):
        box(x, Y[i], W, H, t, fc, ec, size=6.5)
        if i < len(texts) - 1:
            arrow(x, Y[i] - H / 2, x, Y[i + 1] + H / 2, ec)


stage_boxes("A", [
    "Differential expression\n60 tumours vs 334 GTEx\n(PyDESeq2)",
    "4,006 DEG anchors\n(3,992 after alignment)",
    "No survival data used;\nIDH-WT-only reruns\nkept >97% of anchors",
])
stage_boxes("B", [
    "STRING-constrained\nedge universe: 90,959\nedges, 13,039 genes",
    "Healthy reference\ncorrelations \u03c1 (GTEx)",
    "Reference network:\ntop 20% |\u03c1|\n18,192 edges",
])
stage_boxes("C", [
    "GDC \u2192 recount3\ncalibration\n(60 bridge tumours)",
    "Add-one-in sPCC\n(LIONESS), N = 334;\nedge = f(z\u1d62, z\u2c7c, \u03c1\u1d62\u2c7c)",
    "eNESH per anchor (M3)",
    "Matched baselines:\nexpression (M2), R1,\nR2, R2b",
])
stage_boxes("D", [
    "Event-stratified 80:20 split\ndevelopment n = 115\ninternal partition n = 29",
    "M1 clinical; M2, M3,\nR1\u2013R2b; M4\u2013M6\nstacked (out-of-fold)",
    "Nested CV (3\u00d75 outer,\n3-fold inner);\nRSF / CoxNet / GBSA",
    "Harrell's & Uno's C, AUC(t),\nIBS; 100 alternative splits;\nclinical positive control",
])

# inputs -> stages
for x0, x1 in ((10, 11), (30, 15), (30, 34), (50, 40), (70, 60), (70, 85)):
    arrow(x0, 90 - 3.75, x1, 78 + 2.75, DATA_EC)

# ------------------------------------------------------------------ external evaluation
box(48.5, 15, 95, 9,
    "External evaluation in CGGA (n = 104): frozen TCGA models and feature definitions; control-anchored\n"
    "location shift (20 CGGA non-glioma brain samples); networks rebuilt against the same GTEx reference;\n"
    "no CGGA outcomes used for selection, fitting or tuning; availability-adapted when a gene or KPS was missing",
    EXT_FC, EXT_EC, size=6.5)
arrow(cols["D"], Y[3] - H / 2, cols["D"], 15 + 4.5, STAGE["D"][0])
ax.plot([101, 101], [90, 15], color=EXT_EC, lw=0.9)          # CGGA routed down the right edge
ax.plot([90 + 9, 101], [90, 90], color=EXT_EC, lw=0.9)
arrow(101, 15, 96, 15, EXT_EC)

ax.text(50, 6.5,
        "sPCC, single-sample Pearson correlation; eNESH, per-anchor rewiring score; R1, gene deviation |z|; "
        "R2, neighbour deviation;\nR2b, gene \u00d7 neighbour deviation; AUC(t), time-dependent AUC; "
        "IBS, integrated Brier score.",
        ha="center", va="center", fontsize=6.0, color="#444444")

fig.tight_layout()
fig.savefig(OUT / "Figure1.png", dpi=600, bbox_inches="tight")
print("Saved:", OUT / "Figure1.png")

### Figure 2 -- patient flow and cohort assembly.

In [ ]:
"""
Figure 2 -- patient flow and cohort assembly.

Run locally:  python make_figure2_flow.py
Output:       Downloads/TCGA_GBM_STAGE_D_SURVIVAL/FIGURES/Figure2.png (+ .pdf)

TCGA and GTEx counts are the verified numbers from the manuscript.
CGGA counts are recomputed step by step from the CGGA clinical file, so the
boxes match the data exactly. The script prints every count; check them.
"""
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pandas as pd

DOWNLOADS = BASE
OUT_DIR = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL" / "FIGURES"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CGGA_CLINICAL_PATH = (DOWNLOADS / "CGGA.mRNAseq_693_clinical.20200506"
                      / "CGGA.mRNAseq_693_clinical.20200506.txt")

# ---------------------------------------------------------------- CGGA counts
cg = {}
if CGGA_CLINICAL_PATH.exists():
    c = pd.read_csv(CGGA_CLINICAL_PATH, sep="\t")
    cg["total"] = len(c)
    m_gbm = c["Histology"] == "GBM"
    m_pri = m_gbm & (c["PRS_type"] == "Primary")
    m_wt = m_pri & (c["IDH_mutation_status"] == "Wildtype")
    cg["not_gbm"] = int((~m_gbm).sum())
    cg["not_primary"] = int((m_gbm & ~m_pri).sum())
    cg["not_wt"] = int((m_pri & ~m_wt).sum())
    cg["eligible_dx"] = int(m_wt.sum())
    d = c[m_wt].copy()
    os_ = pd.to_numeric(d["OS"], errors="coerce")
    ev = pd.to_numeric(d["Censor (alive=0; dead=1)"], errors="coerce")
    age = pd.to_numeric(d["Age"], errors="coerce")
    bad_surv = os_.isna() | ev.isna() | (os_ <= 0)
    bad_age = ~bad_surv & ~(age >= 18)
    cg["bad_surv"] = int(bad_surv.sum())
    cg["bad_age"] = int(bad_age.sum())
    keep = ~bad_surv & ~bad_age
    cg["final"] = int(keep.sum())
    cg["deaths"] = int(ev[keep].sum())
    print("CGGA counts:", cg)
    if cg["final"] != 104:
        print("WARNING: final CGGA n is", cg["final"], "not 104 -- check the CGGA filters.")
else:
    print("CGGA clinical file not found -- using manuscript counts.")
    cg = dict(total=693, not_gbm=None, eligible_dx=109, bad_surv=5, bad_age=0,
              final=104, deaths=92)

# ---------------------------------------------------------------- drawing
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7.5})
fig, ax = plt.subplots(figsize=(7.4, 7.6))
ax.set_xlim(0, 100); ax.set_ylim(6, 100); ax.axis("off")

MAIN = "#E8EEF6"; MAIN_E = "#3A5A80"
EXC = "#F7F2EA"; EXC_E = "#9A7B4F"
REF = "#EAF3EC"; REF_E = "#3F7A4E"
END = "#D6E2F0"


def box(x, y, w, h, text, fc=MAIN, ec=MAIN_E, bold=False, size=7.3):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                boxstyle="round,pad=0.2,rounding_size=0.8",
                                fc=fc, ec=ec, lw=0.9))
    ax.text(x, y, text, ha="center", va="center", fontsize=size,
            fontweight="bold" if bold else "normal", linespacing=1.25)


def arrow(x0, y0, x1, y1, color="#333333"):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", lw=0.9, color=color,
                                shrinkA=0, shrinkB=0))


def excl(xm, y, xe, w, h, text):
    arrow(xm, y, xe - w / 2, y, EXC_E)
    box(xe, y, w, h, text, fc=EXC, ec=EXC_E, size=6.6)


def header(x, y, text, color="#222222"):
    ax.text(x, y, text, ha="center", va="center", fontsize=9,
            fontweight="bold", color=color)


H = 5.2          # main box height
W = 21           # main box width
EW = 21          # exclusion box width

# ================= A. TCGA survival cohort (left half, top)
header(25, 98.5, "A  TCGA survival cohort")
mx, ex = 12.5, 36.5
y = [93, 82, 71, 60]
box(mx, y[0], W, H, "TCGA-GBM patients with\nclinical records\nn = 541")
box(mx, y[1], W, H, "Primary-tumour molecular\nrecord in GDC\nn = 259")
box(mx, y[2], W, H, "Confirmed IDH-wildtype\nn = 213")
box(mx, y[3], W, H, "Survival cohort\nn = 144 (111 deaths)", fc=END, bold=True)
for a, b in zip(y[:-1], y[1:]):
    arrow(mx, a - H / 2, mx, b + H / 2)
excl(mx, (y[0] + y[1]) / 2, ex, EW, 4.6, "No primary-tumour\nmolecular record  n = 282")
excl(mx, (y[1] + y[2]) / 2, ex, EW, 4.6, "IDH-mutant  n = 19\nIDH unresolved  n = 27")
excl(mx, (y[2] + y[3]) / 2, ex, EW, 6.0,
     "No STAR-Counts file in\nretrieved manifest  n = 69\n(35 used as discovery tumours)")
# split
ys = 47
ax.plot([mx, mx], [y[3] - H / 2, 53], color="#333333", lw=0.9)
ax.plot([6, 19], [53, 53], color="#333333", lw=0.9)
arrow(6, 53, 6, ys + 2.8)
arrow(19, 53, 19, ys + 2.8)
ax.text(26, 53, "80:20, stratified\nby event, seed 42", ha="left", va="center",
        fontsize=6.4, style="italic", color="#444444")
box(6, ys, 11, 5.6, "Development\nn = 115")
box(19, ys, 11, 5.6, "Internal test\nn = 29")

# ================= B. CGGA (right half, top)
header(76, 98.5, "B  CGGA external cohort")
cx, cex = 64, 88
box(cx, 93, W, H, f"CGGA mRNAseq_693\nn = {cg['total']}")
box(cx, 77, W, H, f"Primary IDH-wildtype\nGBM (WHO grade IV)\nn = {cg['eligible_dx']}")
box(cx, 60, W, H, f"External validation cohort\nn = {cg['final']} ({cg['deaths']} deaths)",
    fc=END, bold=True)
arrow(cx, 93 - H / 2, cx, 77 + H / 2)
arrow(cx, 77 - H / 2, cx, 60 + H / 2)
if cg.get("not_gbm") is not None:
    t1 = (f"Not GBM  n = {cg['not_gbm']}\nRecurrent/secondary  n = {cg['not_primary']}\n"
          f"IDH-mutant or unknown  n = {cg['not_wt']}")
    h1 = 6.4
else:
    t1 = f"Not primary IDH-wildtype\nGBM  n = {cg['total'] - cg['eligible_dx']}"
    h1 = 4.6
t2 = f"Missing or invalid\nsurvival  n = {cg['bad_surv']}"
if cg["bad_age"]:
    t2 += f"\nAge < 18  n = {cg['bad_age']}"
excl(cx, 85, cex, EW, h1, t1)
excl(cx, 68.5, cex, EW, 4.6, t2)
ax.text(cx, 50.5, "Frozen TCGA models applied;\nno CGGA outcomes used for\nselection, fitting or tuning",
        ha="center", va="center", fontsize=6.4, style="italic", color="#444444")

ax.plot([2, 98], [41, 41], color="#BBBBBB", lw=0.6)

# ================= C. Discovery and healthy reference (bottom)
header(50, 37.5, "C  Feature discovery and healthy reference (recount3 / GENCODE v26)", REF_E)
box(17, 27, 28, 9.5,
    "Discovery tumours (TCGA-GBM)\nn = 60, patient-disjoint from\nthe survival cohort\n"
    "IDH-WT 39, mutant 8, unresolved 4,\nno clinical record 9", fc=REF, ec=REF_E, size=6.8)
box(83, 32, 26, H, "GTEx cortex + frontal cortex (BA9)\nn = 510 samples", fc=REF, ec=REF_E, size=6.8)
box(83, 20, 26, H, "One sample per donor\n(shorter ischaemic time)\nn = 334 donors",
    fc=REF, ec=REF_E, size=6.8)
arrow(83, 32 - H / 2, 83, 20 + H / 2, REF_E)
box(50, 15, 30, 13,
    "Uses (no survival data)\n"
    "\u2022 Differential expression, 60 vs 334\n   \u2192 disease anchors\n"
    "\u2022 Healthy co-expression reference (334)\n"
    "\u2022 GDC\u2192recount3 calibration bridge (60)",
    fc="white", ec=REF_E, size=6.8)
arrow(31, 25, 35, 18, REF_E)
arrow(70, 20, 65, 17, REF_E)

fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(OUT_DIR / f"Figure2.{ext}", dpi=600, bbox_inches="tight")
print("Saved:", OUT_DIR / "Figure2.png")

### Figure 3 — expression vs eNESH correlation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Figure 3 -- gene-wise correlation between expression and eNESH (development cohort)
STAGE_D = BASE / "TCGA_GBM_STAGE_D_SURVIVAL"
SPLIT_FILE = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
EXPRESSION_FILE = STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv"
ENESH_FILE = STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv"
OUTPUT_DIR = STAGE_D / "FIGURES"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### 2. LOAD DATA

In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================

split_df = pd.read_csv(SPLIT_FILE)

expression = pd.read_csv(
    EXPRESSION_FILE,
    index_col=0
)

enesh = pd.read_csv(
    ENESH_FILE,
    index_col=0
)

print("Split shape:", split_df.shape)
print("Expression shape:", expression.shape)
print("eNESH shape:", enesh.shape)

print("\nSplit columns:")
print(split_df.columns.tolist())

print("\nExpression first columns:")
print(expression.columns[:5].tolist())

print("\neNESH first columns:")
print(enesh.columns[:5].tolist())

### 3. ALIGN TCGA DEVELOPMENT COHORT

In [ ]:
# ============================================================
# 3. ALIGN TCGA DEVELOPMENT COHORT
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Make IDs comparable
split_df["patient_id"] = split_df["patient_id"].astype(str)
expression.index = expression.index.astype(str)
enesh.index = enesh.index.astype(str)

print("Split labels:")
print(split_df["split"].value_counts())

# Development patients
development_ids = (
    split_df.loc[
        split_df["split"].astype(str).str.lower().str.contains("dev"),
        "patient_id"
    ]
    .astype(str)
    .tolist()
)

print("\nDevelopment patients:", len(development_ids))

assert len(development_ids) == 115, (
    f"Expected 115 development patients, found {len(development_ids)}"
)

# Check that all development patients are present
missing_expr = [p for p in development_ids if p not in expression.index]
missing_enesh = [p for p in development_ids if p not in enesh.index]

print("Missing from expression:", len(missing_expr))
print("Missing from eNESH:", len(missing_enesh))

assert len(missing_expr) == 0
assert len(missing_enesh) == 0

# Restrict and align
expression_dev = expression.loc[development_ids].copy()
enesh_dev = enesh.loc[development_ids].copy()

# Check exact same genes
assert expression_dev.columns.equals(enesh_dev.columns)

genes = expression_dev.columns.tolist()

print("Matched genes:", len(genes))

assert len(genes) == 3669

### 4. GENE-WISE EXPRESSION–eNESH PEARSON CORRELATION

In [ ]:
# ============================================================
# 4. GENE-WISE EXPRESSION–eNESH PEARSON CORRELATION
# ============================================================

results = []

for gene in genes:

    x = pd.to_numeric(
        expression_dev[gene],
        errors="coerce"
    ).to_numpy(dtype=float)

    y = pd.to_numeric(
        enesh_dev[gene],
        errors="coerce"
    ).to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    # Cannot calculate correlation if feature is constant
    if (
        len(x) < 3
        or np.std(x) <= 1e-12
        or np.std(y) <= 1e-12
    ):
        r = np.nan

    else:
        r = pearsonr(x, y).statistic

    results.append(
        {
            "gene": gene,
            "n": len(x),
            "pearson_r": r,
            "abs_pearson_r": abs(r) if np.isfinite(r) else np.nan
        }
    )

corr_df = pd.DataFrame(results)

estimable = corr_df.dropna(
    subset=["pearson_r"]
).copy()

print("=" * 60)
print("GENE-WISE PEARSON CORRELATION")
print("=" * 60)

print("Total matched genes :", len(corr_df))
print("Estimable genes     :", len(estimable))
print("Non-estimable genes :", len(corr_df) - len(estimable))

### 5. SUMMARY STATISTICS

In [ ]:
# ============================================================
# 5. SUMMARY STATISTICS
# ============================================================

r = estimable["pearson_r"].to_numpy()
abs_r = estimable["abs_pearson_r"].to_numpy()

median_signed = np.median(r)

q1_signed = np.percentile(r, 25)
q3_signed = np.percentile(r, 75)

median_abs = np.median(abs_r)

n_lt_030 = np.sum(abs_r < 0.30)
n_ge_050 = np.sum(abs_r >= 0.50)

pct_lt_030 = 100 * n_lt_030 / len(abs_r)
pct_ge_050 = 100 * n_ge_050 / len(abs_r)

print("\nSUMMARY")
print("-" * 60)

print(f"Median signed Pearson r : {median_signed:.6f}")
print(
    f"Signed Pearson IQR      : "
    f"{q1_signed:.6f} to {q3_signed:.6f}"
)

print(f"Median absolute |r|     : {median_abs:.6f}")

print(
    f"|r| < 0.30             : "
    f"{n_lt_030} ({pct_lt_030:.1f}%)"
)

print(
    f"|r| >= 0.50            : "
    f"{n_ge_050} ({pct_ge_050:.1f}%)"
)

### 6. MAIN MANUSCRIPT FIGURE

In [ ]:
# ============================================================
# 6. MAIN MANUSCRIPT FIGURE
# Distribution of absolute Pearson correlations
# ============================================================

fig, ax = plt.subplots(figsize=(8.5, 5.8))

# Histogram
ax.hist(
    abs_r,
    bins=np.linspace(0, 1, 31),
    color="#4C78A8",
    edgecolor="white",
    linewidth=0.8,
    alpha=0.90
)

# Threshold/reference lines
ax.axvline(
    0.30,
    color="#E45756",
    linestyle="--",
    linewidth=2,
    label=r"$|r| = 0.30$"
)

ax.axvline(
    0.50,
    color="#F2A541",
    linestyle="--",
    linewidth=2,
    label=r"$|r| = 0.50$"
)

ax.axvline(
    median_abs,
    color="#54A24B",
    linestyle="-",
    linewidth=2.5,
    label=fr"Median $|r| = {median_abs:.3f}$"
)

# Summary box
summary_text = (
    f"Evaluable genes = {len(estimable):,}\n"
    f"Median |r| = {median_abs:.3f}\n"
    f"|r| < 0.30 = {pct_lt_030:.1f}%\n"
    f"|r| ≥ 0.50 = {pct_ge_050:.1f}%"
)

ax.text(
    0.97,
    0.95,
    summary_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10.5,
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        edgecolor="#BFBFBF",
        alpha=0.95
    )
)

# Labels
ax.set_xlabel(
    r"Absolute Pearson correlation between expression and eNESH, $|r|$",
    fontsize=12
)

ax.set_ylabel(
    "Number of genes",
    fontsize=12
)

ax.set_title(
    "Gene-wise association between expression abundance and eNESH\n"
    "in the TCGA development cohort",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlim(0, 1)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.legend(
    frameon=False,
    fontsize=10,
    loc="upper left"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# Save
png_path = OUTPUT_DIR / "FIGURE_EXPRESSION_ENESH_ABSOLUTE_PEARSON.png"
pdf_path = OUTPUT_DIR / "FIGURE_EXPRESSION_ENESH_ABSOLUTE_PEARSON.pdf"

plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(png_path)
print(pdf_path)

### Figure 5 -- discrimination of all representations, with deviation baselines.

In [ ]:
"""Figure 5 -- discrimination of all representations, with deviation baselines.
Values are taken from Table 2 (tab:main_results). Run: python make_figure5.py"""
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# name, group, devCV mean, devCV SD, internal C, lo, hi, CGGA C, lo, hi, CGGA adapted?
rows = [
    ("M1 Clinical",               "single", 0.612, 0.081, 0.510, 0.368, 0.641, 0.523, 0.452, 0.589, True),
    ("M2 Expression",             "single", 0.510, 0.079, 0.643, 0.480, 0.795, 0.489, 0.417, 0.570, False),
    ("M3 eNESH rewiring",         "single", 0.473, 0.080, 0.640, 0.497, 0.777, 0.543, 0.478, 0.607, True),
    ("R1 Gene deviation",         "base",   0.484, 0.078, 0.624, 0.463, 0.792, 0.473, 0.408, 0.547, False),
    ("R2 Neighbour deviation",    "base",   0.506, 0.081, 0.596, 0.436, 0.741, 0.504, 0.436, 0.577, False),
    ("R2b Gene × neighbour", "base",   0.508, 0.075, 0.506, 0.362, 0.624, 0.487, 0.423, 0.549, True),
    ("M4 Clin + expr",            "stack",  0.566, 0.114, 0.500, None,  None,  0.472, 0.406, 0.539, True),
    ("M5 Clin + rewiring",        "stack",  0.573, 0.108, 0.500, None,  None,  0.472, 0.406, 0.539, True),
    ("M6 Clin + expr + rewiring", "stack",  0.584, 0.101, 0.500, None,  None,  0.472, 0.406, 0.539, True),
]
COL = {"single": "#1F5A96", "base": "#C1661C", "stack": "#5E5E5E"}
GROUP_LABEL = {"single": "Single representations", "base": "Deviation baselines",
               "stack": "Stacked (out-of-fold)"}

# y positions with a gap between groups
ypos, y, prev = [], 0.0, None
for r in rows:
    if prev is not None and r[1] != prev:
        y += 0.7
    ypos.append(y); y += 1; prev = r[1]
ypos = [max(ypos) - v for v in ypos]

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8,
                     "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.9), sharey=True)
titles = ["A  Development CV\n(mean ± SD, 15 outer folds)",
          "B  Internal partition, n = 29\n(C, 95% bootstrap CI)",
          "C  External CGGA, n = 104\n(C, 95% bootstrap CI)"]

for k, ax in enumerate(axes):
    ax.axvline(0.5, color="#888888", ls="--", lw=0.8, zorder=0)
    for r, yy in zip(rows, ypos):
        c = COL[r[1]]
        if k == 0:
            est, lo, hi, adapted = r[2], r[2] - r[3], r[2] + r[3], False
        elif k == 1:
            est, lo, hi, adapted = r[4], r[5], r[6], False
        else:
            est, lo, hi, adapted = r[7], r[8], r[9], r[10]
        if lo is not None:
            ax.plot([lo, hi], [yy, yy], color=c, lw=1.4, solid_capstyle="butt")
        ax.plot(est, yy, marker="o", ms=5.5, mec=c, mew=1.3,
                mfc="white" if adapted else c, zorder=3)
        ax.text(0.905, yy, f"{est:.3f}", va="center", ha="right", fontsize=6.8, color="#333333")
    ax.set_xlim(0.3, 0.91)
    ax.set_xticks([0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
    ax.set_title(titles[k], fontsize=8.2, loc="left", fontweight="bold")
    ax.set_xlabel("Harrell's C-index")
    ax.grid(axis="x", color="#EEEEEE", lw=0.6, zorder=0)
    ax.tick_params(axis="y", length=0)

axes[0].set_yticks(ypos)
axes[0].set_yticklabels([r[0] for r in rows])
for lab, r in zip(axes[0].get_yticklabels(), rows):
    lab.set_color(COL[r[1]])
axes[0].set_ylim(min(ypos) - 0.7, max(ypos) + 0.7)

handles = [Line2D([], [], color=COL[g], marker="o", lw=1.4, ms=5, label=GROUP_LABEL[g])
           for g in ("single", "base", "stack")]
handles.append(Line2D([], [], color="#333333", marker="o", mfc="white", lw=0, ms=5,
                      label="Availability-adapted specification"))
handles.append(Line2D([], [], color="#888888", ls="--", lw=0.8, label="Chance (C = 0.5)"))
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=7.2,
           bbox_to_anchor=(0.55, -0.01))
fig.tight_layout(rect=(0, 0.1, 1, 1))
OUT = BASE / "TCGA_GBM_STAGE_D_SURVIVAL" / "FIGURES"
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "Figure5.png", dpi=600, bbox_inches="tight")
fig.savefig(OUT / "Figure5.pdf", bbox_inches="tight")
print("Saved:", OUT / "Figure5.pdf")